# 01 · Baseline And Protocol

Notebook central para fijar una sola vez el protocolo de evaluación, el esquema de exportación de resultados y los baselines contra los que se comparan LGWR/GWR, SAR, RF + Kriging y GNN.

## Objetivos

- Definir un split reproducible y justo.
- Elegir métricas comunes.
- Crear una función estándar para exportar artefactos por modelo.
- Dejar uno o dos baselines simples para calibrar expectativas.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "01_baseline_and_protocol"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

## Dataset de referencia

In [ ]:
import pandas as pd
import geopandas as gpd

def filling_missing_values(
    df: pd.DataFrame | gpd.GeoDataFrame,
    min_obs_barrio: int = 5
) -> pd.DataFrame | gpd.GeoDataFrame:
    """
    Limpia dataset inmobiliario aplicando reglas basadas en barrio.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame original.
    min_obs_barrio : int
        Mínimo número de observaciones requeridas por barrio.

    Retorna
    -------
    pd.DataFrame
        DataFrame limpio.
    """

    gdf = df.copy()

    # 1️⃣ Filtrar barrios con suficiente soporte
    gdf = gdf[
        gdf.groupby('barrio')['barrio'].transform('size') >= min_obs_barrio
    ].copy()

    # 2️⃣ Antigüedad: media por barrio
    gdf['antiguedad'] = gdf['antiguedad'].fillna(
        gdf.groupby('barrio')['antiguedad'].transform('mean')
    )

    # fallback global
    gdf['antiguedad'] = gdf['antiguedad'].fillna(
        gdf['antiguedad'].mode().iloc[0]
    )

    # 3️⃣ Estado: moda por (barrio, antiguedad)

    gdf['antiguedad_cat'] = (
        gdf['antiguedad']
        .round()
        .astype('Int64')
    )

    estado_moda = (
        gdf
        .dropna(subset=['estado'])
        .groupby(['barrio', 'antiguedad_cat'])['estado']
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    )

    mask_estado = gdf['estado'].isna()

    gdf.loc[mask_estado, 'estado'] = (
        gdf.loc[mask_estado]
        .set_index(['barrio', 'antiguedad_cat'])
        .index
        .map(estado_moda)
    )

    # fallback global
    gdf['estado'] = gdf['estado'].fillna(
        gdf['estado'].mode().iloc[0]
    )

    # 4️⃣ Codificación ordinal de estado

    orden_estado = {
        'Excelente': 5,
        'Muy Bueno': 4,
        'Bueno': 3,
        'Regular': 2,
        'A Refaccionar': 1,
    }

    gdf['estado_num'] = gdf['estado'].map(orden_estado)

    return gdf

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "splits"

train_raw = pd.read_csv(DATA_PATH / "arg_venta_caba_train.csv")
gdf_train = gpd.GeoDataFrame(
    train_raw,
    geometry=gpd.points_from_xy(
        train_raw["longitud"],
        train_raw["latitud"]
    ),
    crs="EPSG:4326"
)

test_raw = pd.read_csv(DATA_PATH / "arg_venta_caba_test.csv")
gdf_test = gpd.GeoDataFrame(
    test_raw,
    geometry=gpd.points_from_xy(
        test_raw["longitud"],
        test_raw["latitud"]
    ),
    crs="EPSG:4326"
)

val_raw = pd.read_csv(DATA_PATH / "arg_venta_caba_val.csv")
gdf_val = gpd.GeoDataFrame(
    val_raw,
    geometry=gpd.points_from_xy(
        val_raw["longitud"],
        val_raw["latitud"]
    ),
    crs="EPSG:4326"
)

target_col = "log_precio"
coord_cols = ["longitud", "latitud"]
feature_cols = [
    "area_m2_total",
    "area_m2_descubierta",
    "ambientes",
    "antiguedad",
    "expensas",
    "estado_num",
]

gdf_train_clean = filling_missing_values(gdf_train)
gdf_test_clean = filling_missing_values(gdf_test)
gdf_val_clean = filling_missing_values(gdf_val)


## Métricas estándar

In [ ]:


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    residuals = y_true - y_pred

    # Evitar división por cero en MAPE
    mask = y_true != 0

    if mask.sum() > 0:
        mape = np.mean(
            np.abs(residuals[mask] / y_true[mask])
        ) * 100
    else:
        mape = np.nan

    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "bias": float(residuals.mean()),
        "median_abs_error": float(np.median(np.abs(residuals))),
        "mape": float(mape),
    }

## Formato común de artefactos

Cada notebook de modelo debería exportar al menos:

- `metrics.json`
- `test_predictions.parquet`
- `residuals.parquet`
- `interpretability.parquet` o un artefacto equivalente
- `run_config.json`

In [ ]:
def export_model_artifacts(model_name, metrics, test_frame, interpretability_frame=None, run_config=None):
    model_dir = PROJECT_ROOT / "notebooks" / "output" / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    (model_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
    test_frame.to_parquet(model_dir / "test_predictions.parquet", index=False)

    residual_cols = [col for col in ["y_true", "y_pred", "residual", "split"] if col in test_frame.columns]
    if residual_cols:
        test_frame[residual_cols + [c for c in [group_col] if c and c in test_frame.columns]].to_parquet(
            model_dir / "residuals.parquet", index=False
        )

    if interpretability_frame is not None:
        interpretability_frame.to_parquet(model_dir / "interpretability.parquet", index=False)

    if run_config is not None:
        (model_dir / "run_config.json").write_text(json.dumps(run_config, indent=2, ensure_ascii=False))

## Baselines tentativos

In [ ]:
# X_train = gdf_train[feature_cols]
# y_train = gdf_train[target_col]
# X_test = gdf_test[feature_cols]
# y_test = gdf_test[target_col]

# dummy = DummyRegressor(strategy="median")
# dummy.fit(X_train, y_train)
# dummy_pred = dummy.predict(X_test)
# regression_metrics(y_test, dummy_pred)

## Checklist para todos los modelos

- Misma transformación del target.
- Mismas observaciones evaluadas.
- Mismo set de features base cuando aplique.
- Misma definición de train/valid/test.
- Exportación de artefactos bajo el formato estándar.